# wiki_search: quick start

`WikiQA` answers questions, checks claims and writes short research briefs using only English Wikipedia, with Claude driving the searches and reads. Every fact is cited to the Wikipedia section it came from.

## Setup (once)

```bash
python3 -m venv .venv
.venv/bin/pip install -r requirements.txt
.venv/bin/python -m ipykernel install --user --name wiki-search --display-name "wiki_search (.venv)"
```

Create a `.env` file in the project root:

```
ANTHROPIC_API_KEY=sk-ant-...
WIKI_USER_AGENT=wiki-search/0.1 (your-contact@example.com)
```

Wikimedia asks API clients to identify themselves; with a contact address you get throttled far less. Then open this notebook with the **wiki_search (.venv)** kernel and run the cells in order. Each question takes roughly 10 seconds to 2 minutes, and a few cents of API usage.

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wiki_search").is_dir())
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from wiki_search import WikiQA

qa = WikiQA()  # Claude Sonnet 4.6; see the README for options


def show(answer):
    """Display an answer or brief, or the clarifying question if the tool asked one."""
    if answer.needs_clarification:
        c = answer.clarification
        display(Markdown(f"**The tool asks:** {c.question}\n\nOptions: {', '.join(c.options)}"))
    else:
        display(Markdown(answer.text))
    for w in answer.warnings:
        print("⚠️", w)
    print(f"({answer.elapsed_s}s)")

## A fact question

The answer comes first, then the cited evidence and sources.

In [2]:
show(qa.ask("How tall is Aconcagua?"))

**Answer:** **6,967.15 metres (22,858.1 feet)**

**Reasoning:**
- Aconcagua, located in Mendoza Province, Argentina, has a summit elevation of 6,967.15 metres (22,858.1 feet), making it the highest mountain outside Asia. [1]

**Sources:**  
[1] Aconcagua (lead) — https://en.wikipedia.org/wiki/Aconcagua

(9.1s)


## A multi-hop question

When one answer is needed to find the next, the tool resolves the chain hop by hop and cites each step: here, which country the islands belong to, then that country's currency.

In [3]:
show(qa.ask("What currency is used in the country the Galápagos Islands belong to?"))

**Answer:** **The United States dollar (USD)**

**Reasoning:**
1. The Galápagos Islands form the Galápagos Province of the Republic of Ecuador. [1]
2. The present currency of Ecuador is the United States dollar; it also mints its own centavo coins. [2]

**Sources:**  
[1] Galápagos Islands (lead) — https://en.wikipedia.org/wiki/Gal%C3%A1pagos_Islands  
[2] Currency of Ecuador (lead) — https://en.wikipedia.org/wiki/Currency_of_Ecuador

(9.6s)


## A research brief

For overviews, histories and comparisons the tool reads several articles and writes a short brief: an overview, sections, how items were chosen, further reading and sources. Before a brief is accepted, each claim is checked against the text of the section it cites. This takes a minute or two.

In [4]:
show(qa.ask("Give me an overview of how the printing press changed Europe."))

## How the Printing Press Changed Europe

Invented by Johannes Gutenberg around 1440, the printing press has been called an 'agent of change' across European society, enabling mass communication and the rapid spread of ideas for the first time. [1][3] Its effects reshaped religion, science, language, and political power simultaneously — accelerating the Reformation and Scientific Revolution, and fuelling the rise of national languages and modern states. [1]

### Rapid Spread
- Gutenberg's press in Mainz was initially the only one in Europe, but by the end of the 15th century around 270 cities were involved in printing; as early as 1480 printers were active in 110 places across Germany, Italy, France, Spain, England, and beyond. [2]
- By 1500 European presses had produced over 20 million copies; that figure grew to an estimated 150–200 million by 1600. [2]

### Literacy and Reading Culture
- Lower costs and wider availability of texts shifted reading from communal oral recitation to private, silent reading — a transition historian Elizabeth Eisenstein calls a move 'from a hearing public to a reading public'. [3]
- Identical printed copies enabled precise citation and stronger authorial identity; features like page numbers, indices, and tables of contents became standard. [3]
- Adult literacy rates rose significantly across Europe in the two centuries after the press was invented, and by 1495–1515 Venetian printer Aldus Manutius had published many Greek classical works in printed editions — a recovery made possible at a speed unachievable by manuscript alone. [3]

### The Protestant Reformation
- Between 1518 and 1520, printers distributed 300,000 copies of Luther's works, making him Europe's most published author around 1520 and spreading his ideas faster than authorities could suppress them. [2]
- Cities with at least one press by 1500 were 52 percentage points more likely to have adopted Protestantism by 1530; the effect was strongest in the Reformation's early phase. [3]
- Where Protestant preaching was banned, pamphlets and clandestine presses became the main channel for Reformed ideas, with Catholic polemic responding in kind; the same dynamic later fuelled the French Wars of Religion, the Dutch Revolt, and the English Civil War. [3]

### Science and Knowledge Sharing
- Scholars previously circulated discoveries through individual manuscript letters; printing let them reach far more people faster, enabling cross-European communities of scientists with shared methods. [3]
- Scientific progress visibly accelerated: Ptolemy's Almagest dominated astronomy for roughly 1,400 years, but Copernicus's rival work was surpassed in under 100 years — a difference Eisenstein attributes largely to print. [3]
- Accurate reproductions of medical and botanical illustrations replaced hand-copied drawings that had degenerated over time, allowing figures like Andreas Vesalius to correct the visual record. [3]

### Politics, Censorship, and Language
- Rulers weaponised print quickly: Thomas Cromwell's 1530s campaign backing Henry VIII's break with Rome was the first European government effort to exploit the press for propaganda. [3]
- Authorities fought back with licensing edicts and Papal Indexes — but printed material crossed borders easily and clandestine presses persisted, making censorship only partially effective. [3]
- As publishing shifted from Latin to regional vernacular languages, printing standardised spelling and syntax; historians Febvre and Martin argue it exercised a greater influence on national languages than any other single factor, contributing to the rise of nationalism. [4]

**How these were chosen:** The subtopics reflect the main impact areas explicitly discussed in the 'Impact' section of the Wikipedia Printing press article, which covers literacy, the Reformation, science, politics, and language, as well as the 'Mass production and spread' section covering the press's geographic rollout. [1][3]

**Further reading:**
- [Johannes Gutenberg](https://en.wikipedia.org/wiki/Johannes_Gutenberg) — Details the inventor's life, workshop, and the specific technical innovations — movable type, ink formula, press mechanism — that made the revolution possible.
- [Propaganda during the Reformation](https://en.wikipedia.org/wiki/Propaganda_during_the_Reformation) — Examines how both Protestants and Catholics weaponised print, with specific pamphlets and key propagandists.
- [Global spread of the printing press](https://en.wikipedia.org/wiki/Global_spread_of_the_printing_press) — Traces how the technology moved beyond Europe to be adopted worldwide by the end of the 19th century.
- [Scientific Revolution](https://en.wikipedia.org/wiki/Scientific_Revolution) — Shows how the wider exchange of ideas enabled by print helped transform natural philosophy into modern science.
- [History of Western typography](https://en.wikipedia.org/wiki/History_of_Western_typography) — Explores how type design and typesetting craft evolved alongside the press across Europe.

**Assumptions & gaps:**
- Most quantitative claims draw on studies cited within Wikipedia: the 60%-faster city growth figure is attributed to economist Jeremiah Dittmar, and the 52-percentage-point Protestantism adoption figure to Jared Rubin.
- The 'hearing public to reading public' framing is attributed in Wikipedia to 'Eisenstein'; the full name Elizabeth Eisenstein appears in the Impact section's opening paragraph.

**Sources:**  
[1] Printing press (lead) — https://en.wikipedia.org/wiki/Printing_press  
[2] Printing press § Mass production and spread in Europe — https://en.wikipedia.org/wiki/Printing_press#Mass_production_and_spread_in_Europe  
[3] Printing press § Impact — https://en.wikipedia.org/wiki/Printing_press#Impact  
[4] Printing press § Language — https://en.wikipedia.org/wiki/Printing_press#Language

⚠️ Claim not backed by its citation (distorted): "Invented by Johannes Gutenberg around 1440, the printing press has been called an 'agent of change' across European society, enabling mass communication and the rapid spread of ideas for the first time.". The cited text: The spread of printing brought about an era of mass communication in European society, allowing information to quickly spread across Europe. Section 3 calls the press an 'agent of change' across 'various societies', not specifically 'European society'.
⚠️ Claim not backed by its citation (distorted): "Adult literacy rates rose significantly across Europe in the two centuries after the press was invented, and by 1495–1515 Venetian printer Aldus Manutius had published many Greek classical works in printed editions — a recovery made possible at a speed unachievable by manuscript alone.". The cited text: The cited text says 'wider availability of texts contributed to increasing adult literacy rates across Europe in the next

## Ambiguous names: when the tool asks, and when it doesn't

If a question could reasonably mean several things, the tool asks which one you meant; call `ask` again with the reply. It doesn't ask when the wording or common usage makes one meaning clear.

**Asks:** there are many Springfields, and nothing in the question says which one.

In [5]:
question = "What is the population of Springfield?"
first = qa.ask(question)
show(first)
if first.needs_clarification:
    show(qa.ask(question, clarification='By "Springfield" I mean Springfield, Illinois.'))

**The tool asks:** There are many places named Springfield! Could you clarify which Springfield you're asking about? For example, are you referring to Springfield, Missouri (the most populous city with that name), Springfield, Illinois (the state capital), Springfield, Massachusetts, Springfield, Ohio, or another Springfield entirely?

Options: Springfield, Missouri, Springfield, Illinois, Springfield, Massachusetts, Springfield, Ohio, Springfield, Oregon, Springfield, Virginia

(8.2s)


**Answer:** **114,394 (2020 U.S. Census); ~208,000 in the metro area**

**Reasoning:**
- Springfield, Illinois had a population of 114,394 at the 2020 United States Census, making it the state's seventh-most populous city. [1]
- Approximately 208,000 residents live in the Springfield metropolitan area, which consists of all of Sangamon and Menard counties. [1]

**Sources:**  
[1] Springfield, Illinois (lead) — https://en.wikipedia.org/wiki/Springfield,_Illinois

(8.3s)


**Doesn't ask:** "Macbeth" is also an opera and a Scottish king, but the play is the obvious reading. The tool answers for it and says which meaning it took.

In [6]:
show(qa.ask("When was Macbeth written?"))

**Answer:** **Around 1606 (first performed 1606; first published 1623)**

**Reasoning:**
- Macbeth is estimated to have been first performed in 1606 and was first published in the Folio of 1623. [1]

**Sources:**  
[1] Macbeth (lead) — https://en.wikipedia.org/wiki/Macbeth

(7.8s)
